# Case 2: Incremental Load — `pedidos`
* **Nome:** Valtair Armendani da Silva
* **RA:** 10781071


**Respostas:**
1. **Incremental em `pedidos` vs `fornecedores_legado`:** `pedidos` possui coluna delta (`data_ultima_atualizacao`) mantida por trigger; `fornecedores_legado` não tem coluna de controle temporal.
2. **Pedido + itens no mesmo JSON:** São gravados na mesma transação na origem (unidade de negócio); mantê-los juntos evita dessincronização na Bronze.
3. **Quando um item pode ser perdido:** Se houver `INSERT`, `UPDATE` ou `DELETE` direto em `itens_pedido` sem atualizar `data_ultima_atualizacao` na tabela `pedidos`.

_________________________________

### 1. Conexão read-only

In [20]:
import json, os
from datetime import datetime, timezone
from pathlib import Path
import boto3, psycopg2
from botocore.exceptions import ClientError
from dotenv import load_dotenv

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(root / ".env")
env = lambda k: os.environ[k].strip()

conn = psycopg2.connect(
    host=env("POSTGRES_HOST"), port=env("POSTGRES_PORT"),
    dbname=env("POSTGRES_DB"), user=env("POSTGRES_USER"), password=env("POSTGRES_PASSWORD"),
    options="-c default_transaction_read_only=on -c timezone=UTC",
)
conn.set_session(isolation_level="REPEATABLE READ", readonly=True)
print("Conectado ao Postgres (READ ONLY | UTC | REPEATABLE READ).")

Conectado ao Postgres (READ ONLY | UTC | REPEATABLE READ).


### 2. Marca d'água

In [21]:
s3 = boto3.client(
    "s3", endpoint_url=env("S3_ENDPOINT_URL"),
    aws_access_key_id=env("MINIO_ROOT_USER"), aws_secret_access_key=env("MINIO_ROOT_PASSWORD"),
)
bucket = env("MINIO_BUCKET_BRONZE")
wm_key = "_controle/watermarks/pedidos.json"

try:
    ultimo_marco = datetime.fromisoformat(json.loads(s3.get_object(Bucket=bucket, Key=wm_key)["Body"].read())["ultimo_marco"])
except ClientError:
    ultimo_marco = datetime.fromisoformat("1970-01-01T00:00:00+00:00")

print("Último marco:", ultimo_marco.isoformat())

Último marco: 2026-11-24T00:50:30.738060+00:00


### 3. Marco de corte

In [22]:
cur = conn.cursor()
cur.execute("SELECT CURRENT_TIMESTAMP;")
marco_corte = cur.fetchone()[0]
print("Marco de corte:", marco_corte.isoformat())

Marco de corte: 2026-11-24T01:13:13.029500+00:00


### 4. Extração com limites bilaterais

In [23]:
cur.execute("""
    SELECT id_pedido, id_cliente, data_pedido, status, total_pedido, data_criacao, data_ultima_atualizacao
    FROM pedidos
    WHERE data_ultima_atualizacao > %s AND data_ultima_atualizacao <= %s
    ORDER BY data_ultima_atualizacao;
""", (ultimo_marco, marco_corte))
pedidos = cur.fetchall()

ids = [str(p[0]) for p in pedidos]
if ids:
    cur.execute("""
        SELECT id_item, id_pedido, id_produto, quantidade, preco_unitario, data_criacao
        FROM itens_pedido WHERE id_pedido = ANY(%s::uuid[]);
    """, (ids,))
    itens = cur.fetchall()
else:
    itens = []

print(f"Extraídos: {len(pedidos)} pedidos e {len(itens)} itens.")

Extraídos: 5 pedidos e 14 itens.


### 5. Montagem do JSON

In [24]:
itens_map = {}
for i in itens:
    itens_map.setdefault(str(i[1]), []).append({
        "id_item": str(i[0]), "id_produto": str(i[2]), "quantidade": int(i[3]),
        "preco_unitario": f"{i[4]:.2f}", "data_criacao": i[5].isoformat(),
    })

docs = [{
    "id_pedido": str(p[0]), "id_cliente": str(p[1]), "data_pedido": p[2].isoformat(),
    "status": p[3], "total_pedido": f"{p[4]:.2f}", "data_criacao": p[5].isoformat(),
    "data_ultima_atualizacao": p[6].isoformat(), "itens": itens_map.get(str(p[0]), []),
} for p in pedidos]

print(f"Documentos montados: {len(docs)}")

Documentos montados: 5


### 6. Validação

In [25]:
cur.execute("SELECT COUNT(*) FROM pedidos WHERE data_ultima_atualizacao > %s AND data_ultima_atualizacao <= %s;", (ultimo_marco, marco_corte))
cnt_ped = cur.fetchone()[0]

if ids:
    cur.execute("SELECT COUNT(*) FROM itens_pedido WHERE id_pedido = ANY(%s::uuid[]);", (ids,))
    cnt_itm = cur.fetchone()[0]
else:
    cnt_itm = 0

cur.close()
conn.commit()
conn.close()

assert len(docs) == cnt_ped and sum(len(d["itens"]) for d in docs) == cnt_itm, "Divergência na contagem!"
print(f"Validação OK: {cnt_ped} pedidos e {cnt_itm} itens.")

Validação OK: 5 pedidos e 14 itens.


### 7. Delta vazio

In [26]:
tem_dados = len(docs) > 0
if not tem_dados:
    print("Delta vazio: nenhum pedido alterado no intervalo. Nenhuma gravação será executada.")
else:
    print(f"Delta com {len(docs)} pedido(s).")

Delta com 5 pedido(s).


### 8. Gravação

In [27]:
if tem_dados:
    now = datetime.now(timezone.utc)
    run_id = now.strftime("%Y%m%dT%H%M%SZ")
    prefix = f"pedidos/ano={now:%Y}/mes={now:%m}/dia={now:%d}/execucao={run_id}"
    key_jsonl = f"{prefix}/pedidos.jsonl"

    body = "\n".join(json.dumps(d, ensure_ascii=False) for d in docs) + "\n"
    s3.put_object(Bucket=bucket, Key=key_jsonl, Body=body.encode("utf-8"))
    s3.put_object(Bucket=bucket, Key=f"{prefix}/_SUCCESS", Body=b"")
    s3.put_object(Bucket=bucket, Key=wm_key, Body=json.dumps({
        "tabela": "pedidos",
        "coluna_delta": "data_ultima_atualizacao",
        "ultimo_marco": marco_corte.isoformat(),
        "run_id": run_id,
    }, indent=2).encode("utf-8"))

    print(f"Gravado em: s3://{bucket}/{prefix}/")

Gravado em: s3://bronze/pedidos/ano=2026/mes=11/dia=24/execucao=20261124T011331Z/


### 9. Verificação

In [28]:
if tem_dados:
    print("Arquivos na pasta:", [o["Key"] for o in s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/")["Contents"]])
    lidos = [json.loads(l) for l in s3.get_object(Bucket=bucket, Key=key_jsonl)["Body"].read().decode().splitlines()]
    print(f"Lidos -> Pedidos: {len(lidos)} | Itens: {sum(len(x['itens']) for x in lidos)}")
    print("\nPrimeiro documento:\n", json.dumps(lidos[0], indent=2, ensure_ascii=False))
    print("\nWatermark atual:\n", s3.get_object(Bucket=bucket, Key=wm_key)["Body"].read().decode())

Arquivos na pasta: ['pedidos/ano=2026/mes=11/dia=24/execucao=20261124T011331Z/_SUCCESS', 'pedidos/ano=2026/mes=11/dia=24/execucao=20261124T011331Z/pedidos.jsonl']
Lidos -> Pedidos: 5 | Itens: 14

Primeiro documento:
 {
  "id_pedido": "64fdda37-ce63-4ae0-83c7-f1d9a3d7ad9b",
  "id_cliente": "f842e163-5455-4290-817c-75c5a87319fe",
  "data_pedido": "2026-07-09T21:56:12.931367+00:00",
  "status": "ENVIADO",
  "total_pedido": "216.90",
  "data_criacao": "2026-07-09T21:56:12.931367+00:00",
  "data_ultima_atualizacao": "2026-11-24T01:12:19.607304+00:00",
  "itens": [
    {
      "id_item": "d5f00f27-1e46-488e-abc0-30e4f7009428",
      "id_produto": "658e54a2-9879-4f35-b471-99869d264599",
      "quantidade": 1,
      "preco_unitario": "216.90",
      "data_criacao": "2026-07-09T21:56:12.931367+00:00"
    }
  ]
}

Watermark atual:
 {
  "tabela": "pedidos",
  "coluna_delta": "data_ultima_atualizacao",
  "ultimo_marco": "2026-11-24T01:13:13.029500+00:00",
  "run_id": "20261124T011331Z"
}
